---
layout: post
courses: { csa: {week: 4} }
categories: [Java, Implementing-2D-Array-Algorithms]
lesson_language: Java
lesson_topic: Implementing-2D-Array-Algorithms
lesson_part: interactive
lesson_type: lesson
toc: true
title: Implementing 2D Array Algorithms
menu: nav/csa_units/csaunit4.html
permalink: /csa/unit_04/4_13
---

# 🚨 3:47 AM — The Sub Call
### AP CSA · Topic 4.13 — Implementing 2D Array Algorithms

Your phone buzzes. It's your teacher.

> "I'm out sick today. I need you to run 1st period. I already told them you know the material — you've got the seating chart, the quiz scores, everything's in the system. Just... don't lose anyone. 🙏"

You're not actually a substitute teacher. You're an AP CSA student who is about to discover something: **a classroom seating chart *is* a 2D array**, and every single task a sub has to do on day one — take attendance, find an empty seat, figure out who needs help, spot the top scorer — is a **2D array algorithm** wearing a disguise.

    "By the end of this notebook you will have written the exact code that runs your classroom. Not a made-up dataset. Not `int[][] arr = { {1,2}, {3,4} }` for no reason. Your actual first-period seating chart.\n",

---

### The one idea that makes 4.13 click

A 2D array is **an array, wearing another array as a trenchcoat.**

```
int[][] chart          <-  the trenchcoat (the outer array: one slot per ROW)
   chart[row]           <-  the array *inside* the trenchcoat at that row
      chart[row][col]   <-  one specific kid, standing at a specific seat
```

Every "2D array algorithm" you're about to learn — traverse it, search it, sum it, find the max — **you already know how to do it.** You did it in Unit 6 with plain 1D arrays. The only new skill in 4.13 is: *do the 1D thing once per row (or once per column).* That's it. That's the whole unit. Everything below is just you proving that to yourself, over and over, until it's automatic.

**By the end of this notebook you will be able to:**
- Traverse a 2D array in row-major and column-major order, and explain *why* the order you pick changes what's easy to compute
- Write nested `for` loops **and** nested enhanced-`for` loops over a 2D array, and know exactly when each one is (and isn't) the right tool
- Search a 2D array for a value and return its location
- Accumulate sums, counts, and averages across rows, columns, and the whole grid
- Track a "best so far" value (and its location) while traversing — the pattern behind every max/min-finding FRQ
- Spot and fix the #1 bug that kills 2D array FRQ points: off-by-one loop bounds


## 🧭 How this notebook works

- **Code cells are real Java**, written the way you'd write it in JShell/IJava (top-level methods, no `public class` wrapper needed to run).
  - **If you have a Java Jupyter kernel installed** (e.g. [IJava](https://github.com/SpencerPark/IJava)), pick it as your kernel in VS Code's top-right kernel picker and run cells top to bottom for real.
  - **If you don't**, that's completely fine — read, write, and reason through every `// TODO`, then paste your finished methods into any `.java` file (repl.it, your IDE, whatever you use for class) to compile and test them. The learning happens in the writing, not the clicking.
- **`// TODO`** marks a blank *you* need to fill in. Everything else is scaffolding — given to you on purpose so you spend your energy on the actual algorithm, not retyping boilerplate.
- **`<details>` blocks** are hidden hints. Try the problem *without* opening them first. Struggling productively for 3–5 minutes before peeking is where the actual learning lives — open them the moment you're stuck, not the moment it gets hard.
- Every chapter uses the **same seating chart**, so your understanding compounds instead of resetting with a new "generic" example every time. By Chapter 9 that grid should feel like your own backyard.


## Chapter 1 — The Chart Itself

Picture your actual classroom: 5 pods of tables (Pod A through Pod E), 6 seats per pod. Every seat either has a kid in it with a quiz score, or it's empty because someone's absent.

That's a grid. Rows = pods. Columns = seat number within the pod. One number per seat. **That is a 2D array.**

```
              Seat1  Seat2  Seat3  Seat4  Seat5  Seat6
Pod A  [ 0]     87     92    ---    76     88     91
Pod B  [ 1]    ---     65     72    89    ---     94
Pod C  [ 2]     90     88     77   ---     82     79
Pod D  [ 3]     68    ---     95    91     84    ---
Pod E  [ 4]     73     86     90    88    ---     77
```

`---` means the seat is empty — in code we'll represent "empty" with the sentinel value `-1`, since a real quiz score is never negative.

### The two numbers that describe *any* 2D array
- `chart.length` → the number of **rows** (how many pods)
- `chart[row].length` → the number of **columns in that specific row** (how many seats in that pod)

For a clean rectangular grid like ours, `chart[row].length` is the same for every row, so you'll often see people write `chart[0].length` to mean "the number of columns." That's a shortcut that works *only* because the grid is rectangular — AP CSA calls a grid where rows can have different lengths a **jagged array**. We won't need one today, but now you'll recognize the term the instant it shows up on the exam.

> ### 🔧 Java Hack — 3 ways to build a 2D array
> ```java
> // 1. Empty grid, fill it in later (all values default to 0)
> int[][] chart = new int[5][6];
>
> // 2. Literal, values known up front — what we're using today
> int[][] chart = {
>     {87, 92, -1, 76, 88, 91},
>     {-1, 65, 72, 89, -1, 94}
> };
>
> // 3. Rows sized individually (this is how you'd build a JAGGED array)
> int[][] jagged = new int[3][];
> jagged[0] = new int[4];
> jagged[1] = new int[2];   // shorter row — perfectly legal
> ```
> The AP exam loves to test whether you know `chart.length` is rows and `chart[row].length` is that row's column count — **not** the other way around. Say it out loud once: *"length is rows, [row].length is columns."* That sentence has saved more FRQ points than almost anything else in this unit.


In [ ]:
import java.util.Arrays;

// This is YOUR 1st period. -1 means the seat is empty (student absent).
// Anything else is that student's most recent quiz score.
int[][] scoreChart = {
    {87, 92, -1, 76, 88, 91},   // Pod A
    {-1, 65, 72, 89, -1, 94},   // Pod B
    {90, 88, 77, -1, 82, 79},   // Pod C
    {68, -1, 95, 91, 84, -1},   // Pod D
    {73, 86, 90, 88, -1, 77}    // Pod E
};

String[] podNames = {"Pod A", "Pod B", "Pod C", "Pod D", "Pod E"};

System.out.println("Grid loaded: " + scoreChart.length + " pods x " + scoreChart[0].length + " seats.");


## Chapter 2 — The Full Sweep (row-major traversal)

First thing any sub does: walk the room and see who's there. Pod by pod, seat by seat — the same order your eyes have read every worksheet since kindergarten: **left to right, top to bottom.** That's called **row-major order**, and it's the default way people traverse a 2D array.

The pattern is always the same shape:
```java
for (int row = 0; row < chart.length; row++) {          // walk each pod...
    for (int col = 0; col < chart[row].length; col++) {  // ...then each seat in it
        // do something with chart[row][col]
    }
}
```

**Your task:** write `printSeatingChart`, which prints every seat in row-major order, formatted like:
```
Pod A, Seat 1: 87
Pod A, Seat 2: 92
Pod A, Seat 3: EMPTY
...
```
Remember seat numbers are 1-indexed for a human sub, even though your loop variable `col` is 0-indexed. And don't forget: `-1` should print as `"EMPTY"`, not `-1` — a sub doesn't care about your sentinel value, they care about a kid who isn't there.


In [ ]:
public static void printSeatingChart(int[][] chart, String[] podNames) {
    for (int row = 0; row < /* TODO */; row++) {
        for (int col = 0; col < /* TODO */; col++) {
            String seatValue = (chart[row][col] == -1) ? "EMPTY" : /* TODO: turn the score into a String */;
            System.out.println(/* TODO: print podNames[row] + ", Seat " + (seat number) + ": " + seatValue */);
        }
    }
}

// Run it:
printSeatingChart(scoreChart, podNames);


<details>
<summary>💡 Hint 1 — the loop bounds</summary>

`row` walks the pods, so it should be bounded by `chart.length`. `col` walks the seats *inside that specific pod*, so it should be bounded by `chart[row].length` (not `chart[0].length` — same value here since the grid is rectangular, but `chart[row].length` is the version that's *always* correct, even for a jagged array).
</details>

<details>
<summary>💡 Hint 2 — the seat number</summary>

`col` is 0-indexed (0, 1, 2, 3, 4, 5) but humans count seats starting at 1. You need `col + 1` in your printed output, not `col`.
</details>

<details>
<summary>✅ Full walkthrough + solution</summary>

```java
public static void printSeatingChart(int[][] chart, String[] podNames) {
    for (int row = 0; row < chart.length; row++) {
        for (int col = 0; col < chart[row].length; col++) {
            String seatValue = (chart[row][col] == -1) ? "EMPTY" : String.valueOf(chart[row][col]);
            System.out.println(podNames[row] + ", Seat " + (col + 1) + ": " + seatValue);
        }
    }
}
```

**Trace it:** outer loop `row = 0` locks us into Pod A. Inner loop then runs `col = 0..5`, printing all 6 seats of Pod A *before the outer loop ever moves to `row = 1`.* That "finish the whole inner loop before advancing the outer loop" behavior is the entire definition of row-major order — internalize that sentence and half of 4.13 is free.

This traversal runs `chart.length * chart[row].length` times total — 5 × 6 = 30 seat-visits. That's **O(rows × columns)**, which for a square-ish grid you'll often see simplified to O(n²). Every algorithm in this notebook has that same cost, because they're all built on this same double loop.
</details>


## Chapter 3 — The Seat-Number Sweep (column-major traversal)

Second period, the assistant principal texts you: *"Seat 4 in every pod keeps failing quizzes — is it near the projector glare? Check it."*

Now you don't care about pods at all. You care about **one seat number, checked across every pod.** That means you need to visit `[0][3]`, then `[1][3]`, then `[2][3]`... — moving down a *column* before moving to the next one. This is **column-major order**, and the only thing that changes from Chapter 2 is **which variable is the outer loop.**

```java
for (int col = 0; col < numCols; col++) {        // pick a seat number...
    for (int row = 0; row < chart.length; row++) { // ...then check it in every pod
        // chart[row][col]
    }
}
```

**Your task:** write `printBySeatNumber`, which prints, for every seat number 1–6, that seat's score in every pod — column-major order. Same output format idea as before, but the *order* the lines print in should now go seat-by-seat instead of pod-by-pod.


In [ ]:
public static void printBySeatNumber(int[][] chart, String[] podNames) {
    int numCols = chart[0].length;
    for (int col = 0; col < /* TODO */; col++) {
        for (int row = 0; row < /* TODO */; row++) {
            String seatValue = (chart[row][col] == -1) ? "EMPTY" : String.valueOf(chart[row][col]);
            System.out.println(/* TODO: print "Seat " + (seat number) + " - " + podNames[row] + ": " + seatValue */);
        }
    }
}

printBySeatNumber(scoreChart, podNames);


<details>
<summary>💡 Hint</summary>

Notice `numCols` is grabbed *once*, outside both loops, using `chart[0].length`. That's safe here because we know the grid is rectangular — every row has the same number of columns. If this were a jagged array, "the number of columns" wouldn't even be a well-defined single number, and column-major traversal would need a completely different strategy. This is exactly why the exam sometimes explicitly tells you a 2D array parameter "is not necessarily rectangular" — that phrase is your signal that column-major tricks like this one are off the table.
</details>

<details>
<summary>✅ Full walkthrough + solution</summary>

```java
public static void printBySeatNumber(int[][] chart, String[] podNames) {
    int numCols = chart[0].length;
    for (int col = 0; col < numCols; col++) {
        for (int row = 0; row < chart.length; row++) {
            String seatValue = (chart[row][col] == -1) ? "EMPTY" : String.valueOf(chart[row][col]);
            System.out.println("Seat " + (col + 1) + " - " + podNames[row] + ": " + seatValue);
        }
    }
}
```

The **only structural change from Chapter 2 is which variable is declared first.** Same two variables, same array accesses, same `chart[row][col]`. That swap is the entire concept of "traversal order" — there's no new syntax to learn, only a new *decision* about which relationship in your data you're trying to expose. Rows first if the question is about pods. Columns first if the question is about seat position. The data doesn't change. The question you're asking it does.
</details>

> ### 🔧 Java Hack — the one-line debug print
> ```java
> import java.util.Arrays;
> System.out.println(Arrays.deepToString(scoreChart));
> ```
> `Arrays.deepToString` prints a full 2D array, nested brackets and all, in one line — perfect for sanity-checking your data without writing a whole traversal method just to look at it. (Plain `Arrays.toString` on a 2D array will betray you: it prints memory addresses like `[I@1b6d3586`, because a 2D array is really an array *of arrays*, and `toString` doesn't know to look inside each one.)


## Chapter 4 — Fast-Forward Mode (nested enhanced `for`)

Sometimes you genuinely don't care *where* a value is — you just need to touch every value once. That's when the enhanced `for` loop earns its keep: less to type, less to get off-by-one on.

The trenchcoat metaphor pays off hard here — because `chart` is an array of `int[]` rows, the outer enhanced loop hands you one full row (a real `int[]`) at a time, and the inner enhanced loop walks *that* row:

```java
for (int[] row : chart) {       // "row" here is one whole int[] — one pod's worth of seats
    for (int seat : row) {      // now walk that one row like a normal 1D array
        // use seat
    }
}
```

**Your task:** write `totalOccupiedScore`, which returns the sum of every real score in the grid (skip `-1`s entirely — don't add them, don't count them) using **only enhanced `for` loops**.

**The catch you need to notice going in:** with this pattern you get the *value*, never the row/column index. If a future task needs you to know *where* something is, enhanced `for` is off the table — you're back to Chapter 2/3-style indexed loops. Knowing which tool to reach for is half the actual skill here.


In [ ]:
public static int totalOccupiedScore(int[][] chart) {
    int total = /* TODO: starting value for a running sum */;
    for (int[] row : chart) {
        for (int seat : /* TODO */) {
            if (seat != -1) {
                /* TODO: add seat to the running total */
            }
        }
    }
    return total;
}

System.out.println("Total occupied score: " + totalOccupiedScore(scoreChart));


<details>
<summary>💡 Hint</summary>

A running sum always starts at the identity value for addition: `0`. This is the exact same accumulator pattern from your 1D `sum` algorithm in Unit 6 — you're just running it once per row instead of once total.
</details>

<details>
<summary>✅ Full walkthrough + solution</summary>

```java
public static int totalOccupiedScore(int[][] chart) {
    int total = 0;
    for (int[] row : chart) {
        for (int seat : row) {
            if (seat != -1) {
                total += seat;
            }
        }
    }
    return total;
}
```

If you did this correctly on the real data, you should get **`1922`** — the sum of every real quiz score, across all 23 occupied seats out of 30 total. (Verify: 5 pods × 6 seats = 30 seats, minus 7 empty ones = 23 filled.) If your number doesn't match, the two most common bugs are (1) accidentally adding the `-1`s in, or (2) initializing `total` to something other than `0`.
</details>

> ### 🔧 Java Hack — enhanced `for` is read-only (by feel)
> ```java
> for (int seat : row) {
>     seat = 0;   // compiles fine... and does absolutely nothing to `row`
> }
> ```
> `seat` is a **copy** of the value at that position, not a window into the array. Mutating it mutates a throwaway local variable, not your data. If a task ever needs you to *change* the grid (curve every score by +5, clear a row, etc.), you must go back to an indexed loop and write `chart[row][col] = ...` directly. Enhanced `for` is for **reading**, indexed `for` is for **writing**. That single sentence resolves a shocking number of "why isn't my array changing" bugs.


## Chapter 5 — Needle in Haystack (search)

A parent calls: *"My kid says they got a 94 on the quiz — can you confirm and tell me where they were sitting?"* Now the index is exactly what you need — you're searching for a **value** and reporting back its **location**.

This is the classic 2D search pattern, and it has one design decision baked into it that the AP exam checks constantly: **what do you return if it's not there?** You can't return a real `{row, col}` pair for "not found" — every possible pair is a real seat. The standard move is a sentinel pair like `{-1, -1}`, exactly the same idea as your `-1` sentinel for an empty seat.

**Your task:** write `findSeat(int[][] chart, int targetScore)`, returning an `int[]` of `{row, col}` for the *first* seat (row-major order) holding that score, or `{-1, -1}` if no seat has it. Once you find it, **stop looking** — don't keep scanning the rest of the grid after you already have your answer. That's not just an optimization; on the exam, "search stops as soon as it finds a match" is an explicit, gradable requirement in FRQ rubrics.


In [ ]:
public static int[] findSeat(int[][] chart, int targetScore) {
    for (int row = 0; row < chart.length; row++) {
        for (int col = 0; col < chart[row].length; col++) {
            if (/* TODO: is this the seat we're looking for? */) {
                return /* TODO: build and return the {row, col} pair */;
            }
        }
    }
    return /* TODO: not-found sentinel */;
}

int[] result = findSeat(scoreChart, 94);
System.out.println("Found at: Pod " + result[0] + ", seat index " + result[1]);


<details>
<summary>💡 Hint — why `return` alone is enough to "stop looking"</summary>

You don't need a `boolean found` flag plus a `break` plus a `break` again to escape both loops. `return` exits the *entire method* immediately, from any depth of nesting, the instant you hit it. That's actually the cleanest possible implementation of "stop as soon as you find it" — simpler than the flag-and-break version most people reach for first.
</details>

<details>
<summary>✅ Full walkthrough + solution</summary>

```java
public static int[] findSeat(int[][] chart, int targetScore) {
    for (int row = 0; row < chart.length; row++) {
        for (int col = 0; col < chart[row].length; col++) {
            if (chart[row][col] == targetScore) {
                return new int[] {row, col};
            }
        }
    }
    return new int[] {-1, -1};
}
```

Searching for `94` should return `{1, 5}` — Pod B, seat index 5 (the 6th seat). Trace through it: row 0 (Pod A) has no 94, so the inner loop finishes all 6 seats without ever returning. Row 1 (Pod B) reaches `col = 5`, sees `chart[1][5] == 94`, and returns immediately — the method never even checks pods C, D, or E. **That early exit is the whole point** — it's the difference between an algorithm that's merely correct and one that's correct *and* efficient, and graders notice the difference.
</details>


## Chapter 6 — The Headcount

Before the bell rings you need three numbers for the front office: how many seats are empty, how many quizzes were actually turned in, and the class average. All three come out of **one traversal** — you don't need three separate loops through the grid, you need three accumulators updated inside the *same* loop.

**Your task:** write `classSummary`, which returns a `double[]` of `{emptyCount, occupiedCount, average}` (yes, mixing an int-ish count into a `double[]` — that's fine, they'll just carry decimal points they don't need). Average = total score ÷ number of occupied seats — **not** ÷ total seats, since empty seats didn't take the quiz.


In [ ]:
public static double[] classSummary(int[][] chart) {
    int emptyCount = 0;
    int occupiedCount = 0;
    int totalScore = 0;

    for (int row = 0; row < chart.length; row++) {
        for (int col = 0; col < chart[row].length; col++) {
            if (chart[row][col] == -1) {
                /* TODO: update the right counter for an empty seat */
            } else {
                /* TODO: update the right counter(s) for an occupied seat */
            }
        }
    }

    double average = /* TODO: guard against dividing by zero if the whole class is absent! */;
    return new double[] {emptyCount, occupiedCount, average};
}

double[] summary = classSummary(scoreChart);
System.out.println("Empty seats: " + summary[0]);
System.out.println("Occupied seats: " + summary[1]);
System.out.println("Class average: " + summary[2]);


<details>
<summary>💡 Hint — the division guard</summary>

If `occupiedCount` were ever `0` (everyone absent — extreme, but the exam loves edge cases), `totalScore / occupiedCount` throws an `ArithmeticException` for `int` division by zero. Check `occupiedCount == 0` first and return `0.0` for the average in that case, *before* you attempt the real division.
</details>

<details>
<summary>✅ Full walkthrough + solution</summary>

```java
public static double[] classSummary(int[][] chart) {
    int emptyCount = 0;
    int occupiedCount = 0;
    int totalScore = 0;

    for (int row = 0; row < chart.length; row++) {
        for (int col = 0; col < chart[row].length; col++) {
            if (chart[row][col] == -1) {
                emptyCount++;
            } else {
                occupiedCount++;
                totalScore += chart[row][col];
            }
        }
    }

    double average = (occupiedCount == 0) ? 0.0 : (double) totalScore / occupiedCount;
    return new double[] {emptyCount, occupiedCount, average};
}
```

Expected numbers on the real chart: **7 empty, 23 occupied, average ≈ 83.57.** Notice the `(double)` cast on `totalScore` — without it, `totalScore / occupiedCount` is `int / int`, which truncates (`1922 / 23` would silently give you `83`, not `83.565...`). This exact truncation bug is one of the most common silent point-losses on FRQs involving averages — the code *runs*, it just gives a subtly wrong number, which is worse than crashing because you might not even notice.
</details>


## Chapter 7 — Pods vs. Seats: Two Questions, One Grid

Time to actually answer the assistant principal's question from Chapter 3, properly: **is there a "bad seat"?** You need two different breakdowns of the exact same data:
- **Per-pod average** (row-based) — is one *pod* underperforming?
- **Per-seat-number average** (column-based) — is one *seat position*, across every pod, underperforming?

Here's the elegant part, and the real insight of this chapter: **you don't need to change your traversal order to get both answers.** You can stay in ordinary row-major order the whole time — the trick is *where you put the accumulator*. A per-pod total resets every time you move to a new row. A per-seat total lives in an array **outside** both loops and keeps building across every row, indexed by `col`.

**Your task:** write `seatNumberAverages`, which returns a `double[]` of length 6 — the average score for seat-number `i` (index `i`) across all pods, ignoring empty seats — using a **single row-major traversal** (no column-major loop needed!).


In [ ]:
public static double[] seatNumberAverages(int[][] chart) {
    int numCols = chart[0].length;
    int[] seatTotals = new int[numCols];   // seatTotals[c] = running sum for seat number c+1
    int[] seatCounts = new int[numCols];   // seatCounts[c] = how many pods had someone in seat c+1

    for (int row = 0; row < chart.length; row++) {
        for (int col = 0; col < chart[row].length; col++) {
            if (chart[row][col] != -1) {
                /* TODO: update seatTotals[col] and seatCounts[col] */
            }
        }
    }

    double[] averages = new double[numCols];
    for (int col = 0; col < numCols; col++) {
        averages[col] = /* TODO: seatTotals[col] / seatCounts[col], watch the divide-by-zero + int-division traps from Ch. 6 */;
    }
    return averages;
}

double[] seatAverages = seatNumberAverages(scoreChart);
for (int i = 0; i < seatAverages.length; i++) {
    System.out.println("Seat " + (i + 1) + " average: " + seatAverages[i]);
}


<details>
<summary>💡 Hint</summary>

`seatTotals` and `seatCounts` are declared **before** the traversal starts and **outside** the row loop, so they persist and accumulate across all 5 pods. Compare that to `totalScore` in Chapter 6, which was a single number accumulating across the *whole* grid — this is the same idea, just spread across 6 buckets (one per column) instead of 1.
</details>

<details>
<summary>✅ Full walkthrough + solution</summary>

```java
public static double[] seatNumberAverages(int[][] chart) {
    int numCols = chart[0].length;
    int[] seatTotals = new int[numCols];
    int[] seatCounts = new int[numCols];

    for (int row = 0; row < chart.length; row++) {
        for (int col = 0; col < chart[row].length; col++) {
            if (chart[row][col] != -1) {
                seatTotals[col] += chart[row][col];
                seatCounts[col]++;
            }
        }
    }

    double[] averages = new double[numCols];
    for (int col = 0; col < numCols; col++) {
        averages[col] = (seatCounts[col] == 0) ? 0.0 : (double) seatTotals[col] / seatCounts[col];
    }
    return averages;
}
```

Run it against the real chart and something jumps out: **Seat 4 has the highest average of any seat position, at 86.0** — nowhere near "failing," the opposite of what the assistant principal suspected. (Seat 1 is actually the lowest, at 79.5.) That's a real finding, produced by a real algorithm, and it's the kind of thing you can only notice once the *code* does the counting instead of you eyeballing a spreadsheet.

This is also the moment to lock in the big transferable idea of Chapter 7: **traversal order and "what you're computing" are independent decisions.** You can compute a column-based statistic while traversing in row-major order, as long as your accumulator is structured to match the question, not the loop. That flexibility is exactly what separates "I memorized the row-major and column-major loop shapes" from "I actually understand 2D arrays" — and it's squarely a 5-level skill.
</details>


## Chapter 8 — The Danger Zone (off-by-one bugs)

Here's the code a *previous* substitute left behind, supposedly to print every score plus a 5-point curve. It compiles. It crashes.

```java
public static void printCurvedScores(int[][] chart) {
    for (int row = 0; row <= chart.length; row++) {
        for (int col = 0; col <= chart[row].length; col++) {
            if (chart[row][col] != -1) {
                System.out.println(chart[row][col] + 5);
            }
        }
    }
}
```

Run the cell below. Read the exception message carefully — it tells you exactly which index and which array size were involved. Then fix the method **in the code cell** so it runs clean.

**Before you fix it, answer this for yourself:** why does `<=` feel so tempting to write here, and what's the actual rule that tells you it's wrong *every single time* for a loop bound built from `.length`?


In [ ]:
public static void printCurvedScores(int[][] chart) {
    for (int row = 0; row <= chart.length; row++) {
        for (int col = 0; col <= chart[row].length; col++) {
            if (chart[row][col] != -1) {
                System.out.println(chart[row][col] + 5);
            }
        }
    }
}

// TODO: run this once broken, read the exception, then fix the method above and re-run.
printCurvedScores(scoreChart);


<details>
<summary>💡 Hint</summary>

Valid indices for an array of length `n` are `0` through `n - 1`. `chart.length` itself is **one past** the last valid row index, and `chart[row].length` is one past the last valid column index. `<=` lets the loop variable reach that one-past value and then try to use it as an index — which is exactly what `ArrayIndexOutOfBoundsException` is telling you happened.
</details>

<details>
<summary>✅ Full walkthrough + solution</summary>

```java
public static void printCurvedScores(int[][] chart) {
    for (int row = 0; row < chart.length; row++) {
        for (int col = 0; col < chart[row].length; col++) {
            if (chart[row][col] != -1) {
                System.out.println(chart[row][col] + 5);
            }
        }
    }
}
```

Both `<=` became `<`. That's the entire fix — the algorithm's *logic* was already correct, only the *bounds* were broken. This is exactly why loop-bound mistakes are so brutal on the actual exam: the grading rubric usually awards a full point just for "correct loop bounds," completely separate from the point for "correct use of array elements." You can nail the hard conceptual part of a question and still lose a guaranteed point to `<=` on muscle memory. Read every bound you write, every time, and ask: *"is this the length, or is this one less than the length?"*
</details>

> ### 🔧 Java Hack — memorize the failure mode, not just the fix
> `ArrayIndexOutOfBoundsException` always names the exact bad index and the array's actual length in its message — e.g. `Index 6 out of bounds for length 6`. **Read that message before you touch the code.** It's not a vague crash; it's the computer telling you precisely which line and which off-by-one to fix. Students who read the exception text fix these bugs in 10 seconds. Students who don't stare at their loops guessing.


## Chapter 9 — Capstone: The SeatFinder Protocol

Last task before the bell: find the top scorer in the entire class, and report exactly where they're sitting — not just the score, the *location*.

This is a different flavor of algorithm than anything above: you're not summing everything, and you're not searching for one known target. You're **tracking the best candidate you've seen so far while you walk the whole grid**, updating your "champion" every time you find something better. This "track the winner while you traverse" pattern is the backbone of nearly every max/min-finding FRQ you will ever see, 2D or otherwise.

**Your task:** write `findTopScorer`, returning an `int[]` of `{row, col, score}` for the highest score in the grid. Start your "best so far" as low as a real score could ever be (`-1`, our empty-seat sentinel, works perfectly here — any real score beats it immediately) so the very first occupied seat you check can legitimately become the new champion.

**Stretch goal (genuinely optional, but this is 5-level thinking):** once you've got `findTopScorer` working, try writing a second method, `nearestEmptySeatToDoor`, that finds the empty seat with the smallest `row + col` (treat Pod A, Seat 1 — index `[0][0]` — as the seat right by the door, and "distance" as `row + col`). Same tracking-the-best pattern, different comparison.


In [ ]:
public static int[] findTopScorer(int[][] chart) {
    int bestScore = -1;
    int bestRow = -1;
    int bestCol = -1;

    for (int row = 0; row < chart.length; row++) {
        for (int col = 0; col < chart[row].length; col++) {
            if (/* TODO: is chart[row][col] a real score AND better than our current champion? */) {
                /* TODO: crown a new champion — update bestScore, bestRow, bestCol */
            }
        }
    }

    return new int[] {bestRow, bestCol, bestScore};
}

int[] top = findTopScorer(scoreChart);
System.out.println("Top scorer: " + podNames[top[0]] + ", seat " + (top[1] + 1) + ", score " + top[2]);

// TODO (stretch goal): write nearestEmptySeatToDoor(int[][] chart) here.


<details>
<summary>💡 Hint</summary>

The condition needs **two things joined with `&&`**: the seat can't be empty (`chart[row][col] != -1`), *and* it has to beat the current champion (`chart[row][col] > bestScore`). Skip the first check and an empty seat's `-1` could never win anyway since real scores are always ≥ 0 — but writing the check explicitly is the safer, more defensible habit, especially if a sentinel value were ever something less obviously "safe" than `-1`.
</details>

<details>
<summary>✅ Full walkthrough + solution</summary>

```java
public static int[] findTopScorer(int[][] chart) {
    int bestScore = -1;
    int bestRow = -1;
    int bestCol = -1;

    for (int row = 0; row < chart.length; row++) {
        for (int col = 0; col < chart[row].length; col++) {
            if (chart[row][col] != -1 && chart[row][col] > bestScore) {
                bestScore = chart[row][col];
                bestRow = row;
                bestCol = col;
            }
        }
    }

    return new int[] {bestRow, bestCol, bestScore};
}
```

On the real chart, the champion is **Pod D, seat 3, with a 95** — and it's crowned at `row = 3`, meaning the algorithm walked through Pods A, B, and C first, briefly crowning 92, then nothing beats it until Pod D's 95 dethrones it. The `94` sitting in Pod B never got a chance — by the time the loop reaches it, `bestScore` is already `92`, and `94 > 92` is true, so it *would* have updated... except wait, walk through it again: does `94` actually update the champion before `95` shows up two rows later? Trace it yourself, seat by seat, and confirm the champion's value after *every single* comparison. That trace-it-by-hand habit — not just trusting that the code "looks right" — is precisely the skill AP CSA's FRQ graders are checking when they read your code line by line.
</details>

**Stretch goal solution** (only look after you've genuinely tried it):

<details>
<summary>✅ nearestEmptySeatToDoor</summary>

```java
public static int[] nearestEmptySeatToDoor(int[][] chart) {
    int bestDistance = Integer.MAX_VALUE;
    int bestRow = -1;
    int bestCol = -1;

    for (int row = 0; row < chart.length; row++) {
        for (int col = 0; col < chart[row].length; col++) {
            if (chart[row][col] == -1) {
                int distance = row + col;
                if (distance < bestDistance) {
                    bestDistance = distance;
                    bestRow = row;
                    bestCol = col;
                }
            }
        }
    }

    return new int[] {bestRow, bestCol};
}
```

Notice the mirror image: `findTopScorer` starts its champion as low as possible (`-1`) and looks for `>`. This method starts its champion as high as possible (`Integer.MAX_VALUE`) and looks for `<`. **Same skeleton, flipped direction.** Once you see that, you've generalized the pattern — you're no longer solving "find the max quiz score," you're solving "find the extreme value of *any* comparable quantity while traversing a grid," which is the actual, transferable skill the exam is testing.
</details>


## 🎯 AP Exam-Ready Checkpoint

Every 2D array FRQ gets graded against a small set of recurring criteria. Before you move on, hold your own code from this notebook up against this checklist — this is genuinely close to how a reader scores your response:

- [ ] **Correct nested loop structure** — two loops, correctly nested (not two separate sequential loops when the task needs every combination of row and column)
- [ ] **Correct loop bounds** — `< length`, never `<= length`; and the *right* length for each loop (`chart.length` for rows, `chart[row].length` for columns — not swapped)
- [ ] **Correct array element access** — `chart[row][col]`, in that order, every time
- [ ] **Correct accumulator initialization** — sums start at `0`, "best so far" trackers start at a value guaranteed to lose immediately, counts start at `0`
- [ ] **Correct handling of the "not found" / edge case** — sentinel returns, divide-by-zero guards, empty-grid behavior
- [ ] **Method signature matches exactly what was asked** — parameter types, return type, and (on the real exam) the *exact* method name and parameter order given in the prompt. Style points don't exist; a perfect algorithm with the wrong signature can lose credit for "does not implement the specified method."

### One more rep — try this cold

Here's an original practice prompt, same universe, no scaffolding. Give yourself 10 focused minutes before opening the model solution.

> Write a method `hasStreak(int[][] chart, int minScore, int streakLength)` that returns `true` if **any single pod (row)** has `streakLength` or more **consecutive** seats (left to right) all scoring `minScore` or higher, treating an empty seat as automatically breaking the streak. Otherwise return `false`.
>
> *(Example: with `minScore = 85` and `streakLength = 3`, a pod of `{87, 92, 76, 88, 91, 91}` does **not** qualify — no 3 back-to-back seats all ≥ 85 — but `{87, 92, 90, 88, 91, 40}` **does**, thanks to seats 1–5.)*


In [ ]:
public static boolean hasStreak(int[][] chart, int minScore, int streakLength) {
    // TODO: it's all yours. Think about what needs to reset every row, versus
    // what needs to reset every time the streak breaks, versus what needs to
    // persist across the entire method call.
    return false;
}

System.out.println(hasStreak(scoreChart, 85, 3));  // expect: true  (Pod D: 91, 84... check it by hand)
System.out.println(hasStreak(scoreChart, 90, 2));  // expect: true  (Pod C: 90, 88 fails at 88<90 -- trace the rest!)


<details>
<summary>✅ Model solution</summary>

```java
public static boolean hasStreak(int[][] chart, int minScore, int streakLength) {
    for (int row = 0; row < chart.length; row++) {
        int currentStreak = 0;   // resets every pod — a streak can't jump between rows
        for (int col = 0; col < chart[row].length; col++) {
            if (chart[row][col] != -1 && chart[row][col] >= minScore) {
                currentStreak++;
                if (currentStreak >= streakLength) {
                    return true;   // found one — no need to keep scanning anything
                }
            } else {
                currentStreak = 0;   // anything else breaks the streak
            }
        }
    }
    return false;   // walked the entire grid, no pod ever qualified
}
```

Three separate variables live at three separate scopes here, and telling them apart is the actual difficulty of this problem: `row`/`col` are loop counters, `currentStreak` resets **every row** (declared *inside* the outer loop, *outside* the inner loop), and the early `return true` is the same "stop as soon as you know the answer" instinct from Chapter 5's search. If you declared `currentStreak` outside the outer loop by mistake, a strong finish in Pod A would incorrectly carry over and inflate Pod B's streak — trace that failure mode by hand once so you never make it on exam day.
</details>


## 🔁 Before you close this notebook

The bell rings. Your teacher texts back: *"How'd it go?"*

Answer these for real — out loud, to a study partner, or just in your head. This is the part that actually moves the material from "I did the exercise" into "I own this pattern":

1. Every algorithm you wrote today — traverse, search, sum, average, find-the-max — you'd already learned for **1D arrays** in an earlier unit. What's the *actual* new skill 4.13 added? (If your answer is "nested loops," go deeper — nested loops are the *mechanism*. What's the *idea*?)
2. When would you reach for row-major order versus column-major order versus enhanced `for`, without being told which to use? Say the actual decision rule, not just "it depends."
3. `findTopScorer` and `nearestEmptySeatToDoor` had the exact same skeleton, mirrored. What has to change, and what stays identical, if you were asked to find the *second*-highest score instead of the highest?

**The payoff, stated plainly:** you now can't look at a spreadsheet, a game board, a seating chart, a pixel grid, or a city map without your brain automatically reaching for row-major/column-major/track-the-best. That's not a metaphor anymore — that's just how you see grids now. The trenchcoat is off. It was two arrays the whole time, and now so is everything else that looks like this.

Go run 2nd period. You've got this.
